# TabTransformer 단독 5-Fold OOF 생성

LightGBM과 완전히 분리된 노트북입니다. lightgbm을 전혀 import하지 않으므로
`torch.set_num_threads(1)` 같은 제약 없이 PyTorch가 모든 코어를 정상적으로 사용합니다.

**사전 준비**: `1차_tabtransformer(->).ipynb`를 먼저 끝까지 실행해서
`tt_checkpoints/fold1.pt` ~ `fold5.pt`가 모두 있어야 합니다.

이 노트북의 결과(`blend_cache/oof_tt.npy`)는 `1차_blend_compare.ipynb`에서
LightGBM 결과와 비교/블렌딩하는 데 사용됩니다.

## 1. 라이브러리 및 설정

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")
torch.manual_seed(1004)
device = torch.device("cpu")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5
CHECKPOINT_DIR = "tt_checkpoints"
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

## 2. 데이터 전처리 (main.py / LightGBM 노트북과 동일)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols and c != TARGET_COL]

cat_cardinalities = []
df_le = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col].astype(str))
    cat_cardinalities.append(df_le[col].nunique())

y = df_le[TARGET_COL].values.astype(np.float32)
X_cat = df_le[cat_cols].values.astype(np.int64)
X_num = df_le[num_cols].values.astype(np.float32)

print(f"전처리 완료: 범주형 {X_cat.shape}, 수치형 {X_num.shape}")

전처리 완료: 범주형 (256351, 20), 수치형 (256351, 47)


## 3. TabTransformer 모델 정의 (체크포인트 로드용, 학습 없음)

In [3]:
class TabTransformer(nn.Module):
    def __init__(self, cat_cardinalities, num_numeric, dim=32, depth=3, heads=4, dropout=0.1):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card + 1, dim) for card in cat_cardinalities])
        self.n_cat = len(cat_cardinalities)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.num_norm = nn.LayerNorm(num_numeric)
        mlp_input_dim = self.n_cat * dim + num_numeric
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1),
        )

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        embs = torch.stack(embs, dim=1)
        embs = self.transformer(embs)
        embs_flat = embs.reshape(embs.size(0), -1)
        x_num_norm = self.num_norm(x_num)
        combined = torch.cat([embs_flat, x_num_norm], dim=1)
        return self.mlp(combined).squeeze(-1)

## 4. 5-Fold OOF 생성 (체크포인트에서 추론만)

In [4]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)
oof_tt = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y), start=1):
    checkpoint_path = f"{CHECKPOINT_DIR}/fold{fold}.pt"
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"{checkpoint_path}가 없습니다. 1차_tabtransformer(->).ipynb를 먼저 끝까지 실행하세요."
        )
    ckpt = torch.load(checkpoint_path, map_location=device)

    scaler = StandardScaler()
    scaler.fit(X_num[tr_idx])

    model = TabTransformer(cat_cardinalities, num_numeric=X_num.shape[1]).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    preds = []
    with torch.no_grad():
        for i in range(0, len(val_idx), 4096):
            batch_idx = val_idx[i : i + 4096]
            xb = torch.from_numpy(X_cat[batch_idx]).long().to(device)
            xn = torch.from_numpy(scaler.transform(X_num[batch_idx]).astype(np.float32)).to(device)
            preds.append(torch.sigmoid(model(xb, xn)).cpu().numpy())
    oof_tt[val_idx] = np.concatenate(preds)

    fold_auc = roc_auc_score(y[val_idx], oof_tt[val_idx])
    print(f"Fold {fold}/{N_SPLITS}  TabTransformer AUC: {fold_auc:.5f}")

score_tt = roc_auc_score(y, oof_tt)
print(f"\nTabTransformer 5-Fold 전체 OOF AUC: {score_tt:.5f}")

Fold 1/5  TabTransformer AUC: 0.73866
Fold 2/5  TabTransformer AUC: 0.73497
Fold 3/5  TabTransformer AUC: 0.73413
Fold 4/5  TabTransformer AUC: 0.73339
Fold 5/5  TabTransformer AUC: 0.73640

TabTransformer 5-Fold 전체 OOF AUC: 0.73523


## 5. 결과 저장 (blend_compare 노트북에서 사용)

In [5]:
np.save(f"{CACHE_DIR}/oof_tt.npy", oof_tt)
print("저장 완료: blend_cache/oof_tt.npy")

저장 완료: blend_cache/oof_tt.npy
